# Query Pipeline — COSORA

Busqueda hibrida (Dense + BM25 con RRF), prompt builder y generacion con LLM.

**Prerequisito:** ejecutar `ingestion_pipeline.ipynb` primero.

## 0. Setup

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
!pip install -q chromadb sentence-transformers rank_bm25 transformers accelerate python-dotenv

import os
import chromadb
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
import torch
from transformers import pipeline
from dotenv import load_dotenv

# Cargar variables de entorno desde Drive
env_path = '/content/drive/MyDrive/variablentorno/.env'
if os.path.exists(env_path):
    load_dotenv(env_path)
    print('✅ Variables de entorno cargadas')

✅ Variables de entorno cargadas


In [15]:
# Config
CHROMA_PATH     = '/content/drive/MyDrive/RAG UPC Final project/chroma_db'
COLLECTION_NAME = 'cosora_chunks'
EMBED_MODEL     = 'BSC-LT/MrBERT-es'

# Backend de generacion: "local" o "openai"
GEN_BACKEND = 'openai'
LOCAL_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

# Conectar
client_chroma = chromadb.PersistentClient(path=CHROMA_PATH)
collection    = client_chroma.get_collection(COLLECTION_NAME)
embed_model   = SentenceTransformer(EMBED_MODEL)

print(f'ChromaDB: {collection.count()} chunks')
print(f'Backend: {GEN_BACKEND}')

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: BSC-LT/MrBERT-es
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB: 214 chunks
Backend: openai


## 1. Retrieval hibrido

In [16]:
def dense_search(query, k=20):
    q_vec = embed_model.encode(query).tolist()
    res   = collection.query(query_embeddings=[q_vec], n_results=k)
    return [
        {'text': doc, 'meta': meta, 'rank_dense': i}
        for i, (doc, meta) in enumerate(zip(res['documents'][0], res['metadatas'][0]))
    ]

In [17]:
# Indice BM25 sobre todos los chunks
all_data  = collection.get()
all_docs  = all_data['documents']
all_metas = all_data['metadatas']

tokenized  = [doc.lower().split() for doc in all_docs]
bm25_index = BM25Okapi(tokenized)
print(f'Indice BM25: {len(all_docs)} chunks')


def bm25_search(query, k=20):
    scores  = bm25_index.get_scores(query.lower().split())
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {'text': all_docs[i], 'meta': all_metas[i], 'rank_bm25': rank}
        for rank, i in enumerate(top_idx)
    ]

Indice BM25: 214 chunks


In [18]:
def rrf_fusion(dense, bm25, k=60, top_n=5):
    """Reciprocal Rank Fusion."""
    scores = {}

    for item in dense:
        cid = item['meta']['chunk_id']
        scores.setdefault(cid, {'text': item['text'], 'meta': item['meta'], 'score': 0.0})
        scores[cid]['score'] += 1.0 / (k + item['rank_dense'])

    for item in bm25:
        cid = item['meta']['chunk_id']
        scores.setdefault(cid, {'text': item['text'], 'meta': item['meta'], 'score': 0.0})
        scores[cid]['score'] += 1.0 / (k + item['rank_bm25'])

    ranked = sorted(scores.values(), key=lambda x: x['score'], reverse=True)
    return ranked[:top_n]

## 2. Prompt Builder

In [19]:
def build_prompt(query, chunks):
    context_blocks = []
    for i, chunk in enumerate(chunks, 1):
        doc_id = chunk['meta']['doc_id']
        context_blocks.append(f'[Fragmento {i} - Fuente: {doc_id}]\n{chunk["text"]}')

    context = '\n\n'.join(context_blocks)

    return f"""Eres COSORA, un asistente experto en obra ferroviaria. \
Responde UNICAMENTE con la informacion del contexto proporcionado. \
Si la informacion no esta en el contexto, dilo explicitamente. \
Cita siempre la fuente (nombre del acta). Responde en espanol.

=== CONTEXTO ===
{context}

=== PREGUNTA ===
{query}

=== RESPUESTA ==="""

## 3. Generador LLM

In [20]:
if GEN_BACKEND == 'local':
    print(f'Cargando {LOCAL_MODEL}...')
    generator = pipeline(
        'text-generation',
        model=LOCAL_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map='auto',
    )
    print('Modelo cargado')


def generate_answer(prompt):
    if GEN_BACKEND == 'local':
        output = generator(
            prompt,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=generator.tokenizer.eos_token_id,
        )
        return output[0]['generated_text'][len(prompt):].strip()

    elif GEN_BACKEND == 'openai':
        from openai import OpenAI
        api_key = os.getenv('OPENAI_API_KEY')
        if not api_key:
            raise ValueError("No se encontro OPENAI_API_KEY en el .env. Comprueba la ruta en la celda de Setup.")
        client = OpenAI(api_key=api_key)
        response = client.chat.completions.create(
            model='gpt-4o',
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0.2,
        )
        return response.choices[0].message.content

    raise ValueError(f'Backend desconocido: {GEN_BACKEND}')

## 4. Pipeline completo

In [21]:
def ask_cosora(query, verbose=True):
    dense  = dense_search(query, k=20)
    bm25   = bm25_search(query, k=20)
    chunks = rrf_fusion(dense, bm25, top_n=20)

    prompt = build_prompt(query, chunks)
    answer = generate_answer(prompt)

    if verbose:
        print(f'Query: {query}')
        print(f'\nChunks recuperados ({len(chunks)}):')
        for c in chunks:
            print(f"  - [{c['meta']['doc_id']}] score={c['score']:.4f} | {c['text'][:80]}...")
        print(f'\nRespuesta ({GEN_BACKEND}):\n{answer}')
        print('-' * 70)

    return answer

## 5. Tests

In [22]:
queries_test = [
    'Que se decidio sobre el talud?',
    'Cual es el estado del camino provisional?',
    'Que materiales se aprobaron para el drenaje de muros?',
]

for q in queries_test:
    ask_cosora(q)
    print()

Query: Que se decidio sobre el talud?

Chunks recuperados (20):
  - [252562-DO-AVO-07-V01] score=0.0167 | nción al cliente El pavimento definido en la partida de presupuesto y en la memo...
  - [254275-DO-AVO-23-V01] score=0.0167 | de la ejecución de la obra. 3. TEMAS DE OBRA: 3.1. Montaje pilares estructura me...
  - [254275-DO-AVO-18-V07] score=0.0164 | de la rampa, con un punto de referencia del proyecto, para poder realizar un seg...
  - [252562-DO-AVO-02-V01] score=0.0164 | , de forma que el Director de obra pueda verificar esta documentación técnica y ...
  - [254275-DO-AVO-21-V01] score=0.0161 | os comentarios que se consideren, antes de este plazo indicado. Conforme Empresa...
  - [252562-DO-AVO-04-V01] score=0.0161 | icas de los materiales a utilizar y demás documentación técnica requerida por le...
  - [254275-DO-AVO-17-V07] score=0.0159 | cante y solicitarás las muestras de Deployé del proyecto. ANEJOS: 01. Apertura d...
  - [254275-DO-AVO-24-V01] score=0.0159 | función para